In [18]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1" #è come dire: se un’operazione NON è supportata su MPS, eseguila automaticamente su CPU invece di crashare
import torch
import torch.nn as nn
import lightning as L
import numpy as np
import segmentation_models_pytorch as smp
from lightning.pytorch.loggers import WandbLogger
import wandb
import matplotlib.pyplot as plt
from lightning.pytorch.callbacks import ModelCheckpoint
from new_data.data_loader import train_df, val_df
from PIL import Image
from torchvision import transforms
import io
import torchmetrics
import sys
import math
sys.path.append("external/ACSNet")


In [ ]:
import io
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode


class ACSNetMulticlassDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

        self.img_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.330, 0.330, 0.330],
                std=[0.204, 0.204, 0.204]
            )
        ])

        self.mask_resize = transforms.Resize(
            (256, 256),
            interpolation=InterpolationMode.NEAREST
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        
        image = Image.open(io.BytesIO(row["image"])).convert("RGB")
        image = self.img_transform(image)

        
        mask = Image.open(io.BytesIO(row["mask"])).convert("L")
        mask = self.mask_resize(mask)
        seg_mask = torch.tensor(np.array(mask), dtype=torch.long)

        
        cls_label = torch.tensor(row["risk_class"], dtype=torch.long)

        return image, seg_mask, cls_label

In [20]:
train_dataset = ACSNetMulticlassDataset(train_df)
val_dataset = ACSNetMulticlassDataset(val_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

In [21]:
import sys
sys.path.append("external/ACSNet")

from MyModel import MyModel

In [22]:
import os
os.listdir("external/ACSNet")
print(os.listdir("external/ACSNet"))

['LICENSE', 'MyModel.py', '1-s2.0-S0010482524004037-main.pdf', 'datasets.py', '__pycache__', 'README.md', 'utils.py', 'joint_transforms.py', '.git', 'DCN.py', 'Train_Seg_Cls_5.py']


In [23]:
model = MyModel()
print(model)

MyModel(
  (base_model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_

In [24]:
batch = next(iter(train_loader))

images = batch[0]
masks = batch[1]
labels = batch[2]

out = model(images)

print(type(out))
print(len(out))
print(masks.shape)

for i, o in enumerate(out):
    print(f"Output {i}:", o.shape)

<class 'tuple'>
2
torch.Size([8, 256, 256])
Output 0: torch.Size([8, 2])
Output 1: torch.Size([8, 3, 256, 256])


In [25]:
class DiceLoss(nn.Module):
    def __init__(self):
        super(DiceLoss, self).__init__()
        self.sigmoid = nn.Sigmoid()

    def forward(self, input, target):
        n = target.size(0) #n: dimensione del batch
        smooth = 1e-5 #serve per evitare divisione per zero 

        input = self.sigmoid(input) #trasformi logits in probabilità

        input_flat = input.view(n, -1) #da [B, 1, H, W] a [B, H*N] quindi ogni immagine diventa un vettore 
        target_flat = target.view(n, -1)

        intersection = input_flat * target_flat #prodotto pixel per pixel

        dice = 2 * (intersection.sum(1) + smooth) / ( #con sum sommi tutti i pixel, ottieni quanto le due immagini si sovrappongono
            input_flat.sum(1) + target_flat.sum(1) + smooth
        )

        loss = 1 - dice.sum() / n 
        return loss

In [26]:
# import torchvision
# class LossNet(torch.nn.Module):
#     def __init__(self, resize=True):
#         super().__init__()

#         vgg = torchvision.models.vgg16(pretrained=True).features

#         self.blocks = torch.nn.ModuleList([
#             vgg[:4].eval(),
#             vgg[4:9].eval(),
#             vgg[9:16].eval(),
#             vgg[16:23].eval(),
#         ])

#         for block in self.blocks:
#             for p in block.parameters():
#                 p.requires_grad = False

#         self.resize = resize
#         self.transform = torch.nn.functional.interpolate

#         self.register_buffer(
#             "mean",
#             torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
#         )
#         self.register_buffer(
#             "std",
#             torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
#         )

#     def forward(self, input, target):
#         if input.shape[1] != 3:
#             input = input.repeat(1, 3, 1, 1)
#             target = target.repeat(1, 3, 1, 1)

#         input = (input - self.mean) / self.std
#         target = (target - self.mean) / self.std

#         if self.resize:
#             input = self.transform(input, size=(256, 256), mode="bilinear", align_corners=False)
#             target = self.transform(target, size=(256, 256), mode="bilinear", align_corners=False)

#         loss = 0.0
#         x = input
#         y = target

#         for block in self.blocks:
#             x = block(x)
#             y = block(y)
#             loss += torch.nn.functional.mse_loss(x, y)

#         return loss

In [27]:
class AutomaticWeightedLoss(nn.Module):
    def __init__(self, num=2):
        super(AutomaticWeightedLoss,self).__init__()
        params = torch.ones(num, requires_grad=True)
        self.params = nn.Parameter(params) #self params sono pesi delle loss che inizialmente sono [1.0,1.0] e poi vengono aggiornati durante il training

    def forward(self, *losses):
        loss_sum = 0
        for i, loss in enumerate(losses):
            loss_sum += 0.5 / (self.params[i] ** 2) * loss + torch.log( #L=∑_i​((1/2*σi**2)​*​Li​+log(1+σi**2​))
                1 + self.params[i] ** 2
            )
        return loss_sum

In [28]:
# class LitACSNet(L.LightningModule):
#     def __init__(self, lr=1e-3):
#         super().__init__()
#         self.save_hyperparameters()

#         self.model = MyModel()

#         self.seg_criterion = DiceLoss()

#         cls_weights = torch.tensor([1.397, 0.779], dtype=torch.float32)
#         self.register_buffer("cls_weights", cls_weights)
#         self.cls_criterion = nn.CrossEntropyLoss(weight=self.cls_weights)

#         self.gamma = 0.2
#         self.lambda_cls = 0.3

#         self.lossnet = LossNet(resize=True)

#         # Metriche segmentazione binaria
#         self.train_seg_dice = torchmetrics.classification.BinaryF1Score()
#         self.val_seg_dice = torchmetrics.classification.BinaryF1Score()

#         self.train_seg_iou = torchmetrics.classification.BinaryJaccardIndex()
#         self.val_seg_iou = torchmetrics.classification.BinaryJaccardIndex()

#         # Metriche classificazione binaria
#         self.train_cls_f1 = torchmetrics.classification.BinaryF1Score()
#         self.val_cls_f1 = torchmetrics.classification.BinaryF1Score()

#         self.train_cls_precision = torchmetrics.classification.BinaryPrecision()
#         self.val_cls_precision = torchmetrics.classification.BinaryPrecision()

#         self.train_cls_recall = torchmetrics.classification.BinaryRecall()
#         self.val_cls_recall = torchmetrics.classification.BinaryRecall()

#         self.train_cls_auc = torchmetrics.AUROC(task="binary")
#         self.val_cls_auc = torchmetrics.AUROC(task="binary")

#     def forward(self, x):
#         # MyModel restituisce: classificazione, segmentazione
#         class_logits, seg_logits = self.model(x)
#         return class_logits, seg_logits

#     def training_step(self, batch, batch_idx):
#         images, seg_masks, cls_labels = batch

#         class_logits, seg_logits = self(images)

#         cls_labels = cls_labels.long()

#         seg_masks = (seg_masks > 0).float()

            # if seg_masks.ndim == 3:
            #     seg_masks = seg_masks.unsqueeze(1)

#         dice_loss = self.seg_criterion(seg_logits, seg_masks)

#         seg_probs = torch.sigmoid(seg_logits)
#         lossnet_loss = self.lossnet(seg_probs, seg_masks)

#         seg_loss = dice_loss + self.gamma * lossnet_loss

#         cls_loss = self.cls_criterion(class_logits, cls_labels)

#         loss = self.lambda_cls * cls_loss + (1 - self.lambda_cls) * seg_loss

#         seg_probs = torch.sigmoid(seg_logits)
#         seg_preds = (seg_probs > 0.5).int()

#         cls_probs = torch.softmax(class_logits, dim=1)[:, 1]
#         cls_preds = torch.argmax(class_logits, dim=1)

#         cls_acc = (cls_preds == cls_labels).float().mean()

#         self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
#         self.log("train_seg_loss", seg_loss, on_step=True, on_epoch=True)
#         self.log("train_cls_loss", cls_loss, on_step=True, on_epoch=True)

#         self.log("train_cls_acc", cls_acc, prog_bar=True, on_step=True, on_epoch=True)

#         self.log(
#             "train_seg_dice",
#             self.train_seg_dice(seg_preds, seg_masks.int()),
#             prog_bar=True,
#             on_step=False,
#             on_epoch=True,
#         )

#         self.log(
#             "train_seg_iou",
#             self.train_seg_iou(seg_preds, seg_masks.int()),
#             prog_bar=True,
#             on_step=False,
#             on_epoch=True,
#         )

#         self.log(
#             "train_cls_f1",
#             self.train_cls_f1(cls_preds, cls_labels),
#             prog_bar=True,
#             on_step=False,
#             on_epoch=True,
#         )

#         self.log(
#             "train_cls_precision",
#             self.train_cls_precision(cls_preds, cls_labels),
#             on_step=False,
#             on_epoch=True,
#         )

#         self.log(
#             "train_cls_recall",
#             self.train_cls_recall(cls_preds, cls_labels),
#             on_step=False,
#             on_epoch=True,
#         )

#         self.log(
#             "train_cls_auc",
#             self.train_cls_auc(cls_probs, cls_labels),
#             prog_bar=True,
#             on_step=False,
#             on_epoch=True,
#         )

#         return loss

#     def validation_step(self, batch, batch_idx):
#         images, seg_masks, cls_labels = batch

#         class_logits, seg_logits = self(images)

#         cls_labels = cls_labels.long()

        # seg_masks = (seg_masks > 0).float()

        # if seg_masks.ndim == 3:
        #     seg_masks = seg_masks.unsqueeze(1)



#         dice_loss = self.seg_criterion(seg_logits, seg_masks)

#         seg_probs = torch.sigmoid(seg_logits)
#         lossnet_loss = self.lossnet(seg_probs, seg_masks)

#         seg_loss = dice_loss + self.gamma * lossnet_loss

#         cls_loss = self.cls_criterion(class_logits, cls_labels)

#         loss = self.lambda_cls * cls_loss + (1 - self.lambda_cls) * seg_loss
        
#         seg_preds = (seg_probs > 0.5).int()

#         cls_probs = torch.softmax(class_logits, dim=1)[:, 1]
#         cls_preds = torch.argmax(class_logits, dim=1)

#         cls_acc = (cls_preds == cls_labels).float().mean()

#         self.val_cls_auc.update(cls_probs, cls_labels)
#         self.val_cls_f1.update(cls_preds, cls_labels)
#         self.val_cls_precision.update(cls_preds, cls_labels)
#         self.val_cls_recall.update(cls_preds, cls_labels)

#         self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
#         self.log("val_seg_loss", seg_loss, on_step=False, on_epoch=True)
#         self.log("val_cls_loss", cls_loss, on_step=False, on_epoch=True)

#         self.log("val_cls_acc", cls_acc, prog_bar=True, on_step=False, on_epoch=True)

#         self.log(
#             "val_seg_dice",
#             self.val_seg_dice(seg_preds, seg_masks.int()),
#             prog_bar=True,
#             on_step=False,
#             on_epoch=True,
#         )

#         self.log(
#             "val_seg_iou",
#             self.val_seg_iou(seg_preds, seg_masks.int()),
#             prog_bar=True,
#             on_step=False,
#             on_epoch=True,
#         )

#         if batch_idx == 0 and self.logger is not None:
#             num_images = min(3, images.shape[0])
#             val_examples = []

#             mean = torch.tensor([0.485, 0.456, 0.406], device=images.device).view(3, 1, 1)
#             std = torch.tensor([0.229, 0.224, 0.225], device=images.device).view(3, 1, 1)

#             class_labels_mask = {
#                 0: "background",
#                 1: "tumor",
#             }

#             for i in range(num_images):
#                 image_vis = images[i].detach() * std + mean
#                 image_vis = image_vis.clamp(0, 1)
#                 image_np = image_vis.permute(1, 2, 0).cpu().numpy()

#                 true_mask_np = seg_masks[i, 0].detach().cpu().numpy().astype(np.uint8)
#                 pred_mask_np = seg_preds[i, 0].detach().cpu().numpy().astype(np.uint8)

#                 val_examples.append(
#                     wandb.Image(
#                         image_np,
#                         masks={
#                             "ground_truth": {
#                                 "mask_data": true_mask_np,
#                                 "class_labels": class_labels_mask,
#                             },
#                             "prediction": {
#                                 "mask_data": pred_mask_np,
#                                 "class_labels": class_labels_mask,
#                             },
#                         },
#                         caption=f"Sample {i} | true cls={cls_labels[i].item()} | pred cls={cls_preds[i].item()}"
#                     )
#                 )

#             self.logger.experiment.log({
#                 "val_examples": val_examples,
#                 "global_step": self.global_step,
#             })

#         return loss

#     def on_validation_epoch_end(self):
#         self.log("val_cls_auc", self.val_cls_auc.compute(), prog_bar=True)
#         self.log("val_cls_f1", self.val_cls_f1.compute(), prog_bar=True)
#         self.log("val_cls_precision", self.val_cls_precision.compute())
#         self.log("val_cls_recall", self.val_cls_recall.compute())

#         self.val_cls_auc.reset()
#         self.val_cls_f1.reset()
#         self.val_cls_precision.reset()
#         self.val_cls_recall.reset()

#     def configure_optimizers(self):
#         optimizer = torch.optim.Adam(
#             self.parameters(),
#             lr=self.hparams.lr,
#             weight_decay=1e-4,
#         )

#         scheduler = torch.optim.lr_scheduler.LambdaLR(
#             optimizer,
#             lr_lambda=lambda epoch: (
#                 ((1 + math.cos(epoch * math.pi / self.trainer.max_epochs)) / 2) * 0.9 + 0.1
#             ),
#         )

#         return {
#             "optimizer": optimizer,
#             "lr_scheduler": scheduler,
#         }

In [29]:
class LitACSNet(L.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.model = MyModel()

        self.seg_criterion = nn.CrossEntropyLoss()

        cls_weights = torch.tensor([1.397, 0.779], dtype=torch.float32)
        self.register_buffer("cls_weights", cls_weights)
        self.cls_criterion = nn.CrossEntropyLoss(weight=self.cls_weights)

        # Loss multi-task come nella repo ACSNet
        self.awl = AutomaticWeightedLoss(num=2)

        # Metriche segmentazione binaria
        self.train_seg_dice = torchmetrics.classification.MulticlassF1Score(
            num_classes=3,
            average="macro"
        )
        self.val_seg_dice = torchmetrics.classification.MulticlassF1Score(
            num_classes=3,
            average="macro"
        )

        self.train_seg_iou = torchmetrics.classification.MulticlassJaccardIndex(
            num_classes=3,
            average="macro"
        )
        self.val_seg_iou = torchmetrics.classification.MulticlassJaccardIndex(
            num_classes=3,
            average="macro"
        )

        # Metriche classificazione binaria
        self.train_cls_f1 = torchmetrics.classification.BinaryF1Score()
        self.val_cls_f1 = torchmetrics.classification.BinaryF1Score()

        self.train_cls_precision = torchmetrics.classification.BinaryPrecision()
        self.val_cls_precision = torchmetrics.classification.BinaryPrecision()

        self.train_cls_recall = torchmetrics.classification.BinaryRecall()
        self.val_cls_recall = torchmetrics.classification.BinaryRecall()

        self.train_cls_auc = torchmetrics.AUROC(task="binary")
        self.val_cls_auc = torchmetrics.AUROC(task="binary")

    def forward(self, x):
        # MyModel restituisce: classificazione, segmentazione
        class_logits, seg_logits = self.model(x)
        return class_logits, seg_logits

    def training_step(self, batch, batch_idx):
        images, seg_masks, cls_labels = batch

        class_logits, seg_logits = self(images)

        cls_labels = cls_labels.long()

        seg_masks = seg_masks.long()

        seg_loss = self.seg_criterion(seg_logits, seg_masks)
        cls_loss = self.cls_criterion(class_logits, cls_labels)

        loss = self.awl(seg_loss, cls_loss)

        seg_probs = torch.softmax(seg_logits, dim=1)
        seg_preds = torch.argmax(seg_probs, dim=1)

        cls_probs = torch.softmax(class_logits, dim=1)[:, 1]
        cls_preds = torch.argmax(class_logits, dim=1)

        cls_acc = (cls_preds == cls_labels).float().mean()

        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train_seg_loss", seg_loss, on_step=True, on_epoch=True)
        self.log("train_cls_loss", cls_loss, on_step=True, on_epoch=True)

        self.log("train_cls_acc", cls_acc, prog_bar=True, on_step=True, on_epoch=True)

        self.log(
            "train_seg_dice",
            self.train_seg_dice(seg_preds, seg_masks),
            prog_bar=True,
            on_step=False,
            on_epoch=True,
        )

        self.log(
            "train_seg_iou",
            self.train_seg_iou(seg_preds, seg_masks),
            prog_bar=True,
            on_step=False,
            on_epoch=True,
        )

        self.log(
            "train_cls_f1",
            self.train_cls_f1(cls_preds, cls_labels),
            prog_bar=True,
            on_step=False,
            on_epoch=True,
        )

        self.log(
            "train_cls_precision",
            self.train_cls_precision(cls_preds, cls_labels),
            on_step=False,
            on_epoch=True,
        )

        self.log(
            "train_cls_recall",
            self.train_cls_recall(cls_preds, cls_labels),
            on_step=False,
            on_epoch=True,
        )

        self.log(
            "train_cls_auc",
            self.train_cls_auc(cls_probs, cls_labels),
            prog_bar=True,
            on_step=False,
            on_epoch=True,
        )

        return loss

    def validation_step(self, batch, batch_idx):
        images, seg_masks, cls_labels = batch

        class_logits, seg_logits = self(images)

        cls_labels = cls_labels.long()

        seg_masks = seg_masks.long()

        seg_loss = self.seg_criterion(seg_logits, seg_masks)
        cls_loss = self.cls_criterion(class_logits, cls_labels)

        loss = self.awl(seg_loss, cls_loss)

        seg_probs = torch.softmax(seg_logits, dim=1)
        seg_preds = torch.argmax(seg_probs, dim=1)

        cls_probs = torch.softmax(class_logits, dim=1)[:, 1]
        cls_preds = torch.argmax(class_logits, dim=1)

        cls_acc = (cls_preds == cls_labels).float().mean()

        self.val_cls_auc.update(cls_probs, cls_labels)
        self.val_cls_f1.update(cls_preds, cls_labels)
        self.val_cls_precision.update(cls_preds, cls_labels)
        self.val_cls_recall.update(cls_preds, cls_labels)

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_seg_loss", seg_loss, on_step=False, on_epoch=True)
        self.log("val_cls_loss", cls_loss, on_step=False, on_epoch=True)

        self.log("val_cls_acc", cls_acc, prog_bar=True, on_step=False, on_epoch=True)

        self.log(
            "val_seg_dice",
            self.val_seg_dice(seg_preds, seg_masks),
            prog_bar=True,
            on_step=False,
            on_epoch=True,
        )

        self.log(
            "val_seg_iou",
            self.val_seg_iou(seg_preds, seg_masks),
            prog_bar=True,
            on_step=False,
            on_epoch=True,
        )

        if batch_idx == 0 and self.logger is not None:
            num_images = min(3, images.shape[0])
            val_examples = []

            mean = torch.tensor([0.330, 0.330, 0.330], device=images.device).view(3, 1, 1)
            std = torch.tensor([0.204, 0.204, 0.204], device=images.device).view(3, 1, 1)

            class_labels_mask = {
                    0: "background",
                    1: "benign_mask",
                    2: "malignant_mask",
                }

            for i in range(num_images):
                image_vis = images[i].detach() * std + mean
                image_vis = image_vis.clamp(0, 1)
                image_np = image_vis.permute(1, 2, 0).cpu().numpy()

                true_mask_np = seg_masks[i].detach().cpu().numpy().astype(np.uint8)
                pred_mask_np = seg_preds[i].detach().cpu().numpy().astype(np.uint8)

                val_examples.append(
                    wandb.Image(
                        image_np,
                        masks={
                            "ground_truth": {
                                "mask_data": true_mask_np,
                                "class_labels": class_labels_mask,
                            },
                            "prediction": {
                                "mask_data": pred_mask_np,
                                "class_labels": class_labels_mask,
                            },
                        },
                        caption=f"Sample {i} | true cls={cls_labels[i].item()} | pred cls={cls_preds[i].item()}"
                    )
                )

            self.logger.experiment.log({
                "val_examples": val_examples,
                "global_step": self.global_step,
            })

        return loss

    def on_validation_epoch_end(self):
        self.log("val_cls_auc", self.val_cls_auc.compute(), prog_bar=True)
        self.log("val_cls_f1", self.val_cls_f1.compute(), prog_bar=True)
        self.log("val_cls_precision", self.val_cls_precision.compute())
        self.log("val_cls_recall", self.val_cls_recall.compute())

        self.val_cls_auc.reset()
        self.val_cls_f1.reset()
        self.val_cls_precision.reset()
        self.val_cls_recall.reset()

    def configure_optimizers(self):
        pg = [p for p in self.parameters() if p.requires_grad]

        optimizer = torch.optim.AdamW(
            pg,
            lr=self.hparams.lr,
            weight_decay=1e-4
        )

        return optimizer

In [30]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",   
    mode="min",           
    save_top_k=1
)

In [31]:
def train():
    wandb.init()

    lr = wandb.config.lr

    wandb_logger = WandbLogger(
        project="ACSNet_model",
        log_model=False
    )

    model = LitACSNet(lr=lr)

    trainer = L.Trainer(
        max_epochs=5,
        log_every_n_steps=1,
        logger=wandb_logger,
        callbacks=[checkpoint_callback]
    )

    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

    wandb.finish()

In [32]:
sweep_config = {
    "method": "grid", #here you are only saying to experiment all the 3 lr
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "lr": {"values": [1e-2, 1e-3, 1e-4]}
    }
}

In [33]:
sweep_id = wandb.sweep(sweep_config, project="ACSNet_model") #It creates a true sweep in the W&B account
print(sweep_id)

Create sweep with ID: drgpt3or
Sweep URL: https://wandb.ai/sara-tramontana02-/ACSNet_model/sweeps/drgpt3or
drgpt3or


In [34]:
wandb.agent(sweep_id, function=train, count=3) #with sweep_id you are telling which sweep to use
#in this case 3 runs since I have 3 lr 

wandb: Agent Starting Run: 6naszf8d with config:
wandb: 	lr: 0.01
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/saratramontana/.netrc.


/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ model               │ MyModel                │ 41.2 M │ train │     0 │
│ 1  │ seg_criterion       │ CrossEntropyLoss       │      0 │ train │     0 │
│ 2  │ cls_criterion       │ CrossEntropyLoss       │      0 │ train │     0 │
│ 3  │ awl                 │ AutomaticWeightedLoss  │      2 │ train │     0 │
│ 4  │ train_seg_dice      │ MulticlassF1Score      │      0 │ train │     0 │
│ 5  │ val_seg_dice        │ MulticlassF1Score      │      0 │ train │     0 │
│ 6  │ train_seg_iou       │ MulticlassJaccardIndex │      0 │ train │     0 │
│ 7  │ val_seg_iou         │ MulticlassJaccardIndex │      0 │ train │     0 │
│ 8  │ train_cls_f1        │ BinaryF1Score          │      0 │ train │     0 │
│ 9  │ val_cls_f1          │ BinaryF1Score          │      0 │ train │     0 │
│ 10 │ train_cls_precision │ BinaryPrecision        │      0 │ train │     0 │
│ 11 │ val_cls_precision   │ BinaryPrecision        │      0 │ train │     0 │
│ 12 │ train_cls_recall    │ BinaryRecall           │      0 │ train │     0 │
│ 13 │ val_cls_recall      │ BinaryRecall           │      0 │ train │     0 │
│ 14 │ train_cls_auc       │ BinaryAUROC            │      0 │ train │     0 │
│ 15 │ val_cls_auc         │ BinaryAUROC            │      0 │ train │     0 │
└────┴─────────────────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 41.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 41.2 M                                                                                               
Total estimated model params size (MB): 164                                                                        
Modules in train mode: 266                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

wandb: WARNING Config item 'lr' was locked by 'sweep' (ignored update).


...

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/dat
a_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing 
the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: No negative samples in targets, false positive value should be meaningless. Returning zero tensor in 
false positive score
  warnings.warn(*args, **kwargs)

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/dat
a_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider 
increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=5` reached.


epoch,▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆█████
global_step,▁▂▄▅▇█
train_cls_acc_epoch,▁▇███
train_cls_acc_step,▂▁███▇█▇▇█████▇████████████▇████████████
train_cls_auc,▁██▃▅
train_cls_f1,▁████
train_cls_loss_epoch,█▂▁▂▁
train_cls_loss_step,███▆▆▄▅▆▆▂▂▂▃▃▂▁▁▁▁▁▁▁▂▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▂
train_cls_precision,▁████
train_cls_recall,▁▇███
+17,...
